In [1]:
! pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort

In [1]:
import sys, os
import matplotlib
import time
import pandas as pd
import numpy
import ast
import json
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from sklearn.neighbors import KNeighborsClassifier as knnbase
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.naive_bayes import MultinomialNB as mnb
from sklearn.linear_model import LogisticRegression as LR
from sklearn.naive_bayes import GaussianNB as GNB

from autogluon.tabular import TabularDataset
from autogluon.tabular import TabularPredictor as task
from autogluon.core.utils import infer_problem_type

from sklearn.model_selection import GridSearchCV as GSCV
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler as MMS
from sklearn.preprocessing import StandardScaler as SS

from sklearn.metrics import accuracy_score, hamming_loss, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import multilabel_confusion_matrix as ML_matrix
from sklearn.metrics import precision_recall_fscore_support as score_multi
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from pickle import load, dump

In [2]:
# -------------------------------- HELPERS ------------------------------------------ #
def split_df(Xdata, labels, testsplit=0.3):
	Xtrain,Xtest,ytrain,ytest = train_test_split(Xdata,labels,test_size=testsplit)
	return Xtrain, Xtest, ytrain, ytest

# Rescale values to fit in a range; default: 0-1
def normalize(Xtrain, Xtest):
	scaler = MMS(feature_range=(0,1))
	Xtrainscaled = scaler.fit_transform(Xtrain)
	Xtestscaled = scaler.transform(Xtest)
	return Xtrainscaled, Xtestscaled

# Scale values such that mean = 0, std dev. = 1; Ensures robustness for new data.
def standardize(Xtrain, Xtest):
	ss = SS()
	Xtrainscaled = ss.fit_transform(Xtrain)
	Xtestscaled = ss.transform(Xtest)
	return Xtrainscaled, Xtestscaled, ss

def micro_avg(y_test_multilabel, predictions):
	precision = precision_score(y_test_multilabel, predictions, average='micro')
	recall = recall_score(y_test_multilabel, predictions, average='micro')
	f1 = f1_score(y_test_multilabel, predictions, average='micro')

	print("::Micro-average::")
	print("Precision: {:.4f}, Recall: {:.4f}, F1-measure: {:.4f}".format(precision, recall, f1))
	print("\n\n")
	return precision, recall, f1

def macro_avg(y_test_multilabel, predictions):
	precision = precision_score(y_test_multilabel, predictions, average='macro')
	recall = recall_score(y_test_multilabel, predictions, average='macro')
	f1 = f1_score(y_test_multilabel, predictions, average='macro')

	print("\nMacro-average: ")
	print("Precision: {:.4f}, Recall: {:.4f}, F1-measure: {:.4f}".format(precision, recall, f1))
	return

def per_class_dist(ytest, ypred, classorder):
	perclass = classification_report(ytest, ypred)
	print("Per class classification report: ", perclass)
	precision, recall, fscore, support = score_multi(ytest, ypred, average="micro")
	print('micro-precision: {}'.format(precision))
	print('micro-recall: {}'.format(recall))
	print('micro-fscore: {}'.format(fscore))
	print('support: {}'.format(support))
	#print(classorder)
	return

def output_avg(total, ag_res1, ag_res2, fimp1, fimp2, auto_cmatrix, bestmodel, perf, auc_score, ff):
	print(auto_cmatrix)
	ff.write("-----------------Autogluon----------------\n")
	ff.write("Best model confusion matrix: \n")
	[tn,fp,fn,tp] = auto_cmatrix
	fpr = float(fp/(fp+tn)*100)
	ff.write("TN: "+str(tn)+" FP: "+str(fp)+" FN: "+str(fn)+" TP: "+str(tp)+"\n")
	ff.write("::Model performance on test data::\n")
	ff.write("AUC Score: "+str(auc_score)+"\n")
	ff.write("FPR: "+str(fpr)+"\n")
	ff.write("Best model: "+ bestmodel+" \n")
	ff.write("Performance summary: "+str(perf)+" \n")
	ff.write(str(ag_res1))
	if not fimp1 == None:
		ff.write("*Ft impo*\n")
		ff.write(str(fimp1.head(20))+"\n")
	ff.write("\n::Stacking & Weighted Ensembling of Models::\n")
	ff.write(str(ag_res2))
	if not fimp2 == None:
		ff.write("*Ft impo*\n")
		ff.write(str(fimp2.head(20))+"\n")
	ff.write("--------------------------------------------\n")
	ff.close()
	return

In [3]:
def test_main(xtest, ytest, pred, testdf, traindf, calcftimpo=False):
	modelperf = pred.leaderboard(testdf, silent= True)
	print("[*]Model performance breakdown on Test data:")
	print(modelperf)
	ypred = pred.predict(xtest)
	ypredproba = pred.predict_proba(xtest)
	perf = pred.evaluate_predictions(y_true=ytest, y_pred=ypred, auxiliary_metrics= True)
	print("[*]Predictions: ", ypred)
	print("[*]Confidence in predictions:\n")
	print(pd.DataFrame(ypredproba, columns=pred.class_labels))
	# Each model score
	print("Perf: ", perf)
	print("Getting confusion matrix.....")
	cmatrix = confusion_matrix(ytest, ypred).ravel().tolist()
	print(cmatrix)
	auc_score = roc_auc_score(ytest, ypredproba.iloc[:, 1])
	print("AUC score for best model: ", auc_score)

	if calcftimpo:
		ftimpo = None
		ftimpo = pred.feature_importance(traindf)
		print("Feature Importance on test data: ", ftimpo)
	else:
		ftimpo = None
	return modelperf, ftimpo, cmatrix, ypredproba, perf, auc_score

def test_stack(xtest, ytest, predstack, testdf, traindf, calcftimpo=False):
	ypred = predstack.predict(xtest)
	ypredproba = predstack.predict_proba(xtest)
	perf = predstack.evaluate_predictions(y_true=ytest, y_pred=ypred, auxiliary_metrics= True)
	print("[*]Predictions: ", ypred)
	test_perf = predstack.leaderboard(testdf, silent=True)
	print("$$$$$$$$ RESULT STACKING $$$$$$$$\n", test_perf)
	ftimpo = None
	if calcftimpo:
		ftimpo = predstack.feature_importance(traindf)
		print("Feature Importance on test data: ", ftimpo)
	auc_score = roc_auc_score(ytest, ypredproba.iloc[:, 1])
	cmatrix = confusion_matrix(ytest, ypred).ravel().tolist()
	print("Confusion matrix stacked: ", cmatrix)
	print("AUC using stacked model: ", auc_score)
	return test_perf, ftimpo, cmatrix, auc_score

In [4]:
def train_main(dataf, targetcol):
	agdir = os.getcwd()+'/AGmodels/'
	#dir = agdir+"/"+str(malinst)+"_"+str(hostfts)+"/"
	if not os.path.exists(agdir):
		os.system("mkdir "+agdir)

	predictor = task(label=targetcol, path=agdir, eval_metric='balanced_accuracy').fit(dataf, verbosity=4)
	return predictor

# Multi layer stacking takes predictions of base models and feeds to stack models
# AG will auto choose k= 10 fold cv, n=20 bagging repeats,
# L: 2 layers of models in stack followed by weighted-ensemble (higher weight for the model that performed well);
# Aggregate model predictions based on model weights and produce final prediction
def train_multilayerstacking(traindf, target):
	agdir_stack = os.getcwd()+'/AGmodels/stacked/' #+str(malinst)+"_"+str(hostfts)+"/"
	if not os.path.exists(agdir_stack):
		os.system("mkdir "+agdir_stack)
	predstack = task(label=target, path=agdir_stack, eval_metric='balanced_accuracy').fit(train_data= traindf, auto_stack=True, verbosity=3)
	return predstack

def main_ag(traindf, testdf, targetcol):
	# Displaying dataframe info
	x_test = testdf.iloc[:,:-1].copy()
	y_test = testdf.iloc[:,-1].copy()
	proxy_train = traindf[traindf['target'] == 1].shape
	proxy_test = testdf[testdf['target'] == 1].shape
	normal_train = traindf[traindf['target'] == 0].shape
	normal_test = testdf[testdf['target'] == 0].shape

	time.sleep(2)

  # Training binary classifiers: 8 base models, 2 DL models
	predictor = train_main(traindf, targetcol)
	predstack = train_multilayerstacking(traindf, targetcol)

	# Testing binary classifiers
	print("###################~Testing Trained Models (Never seen PCAPS)~############################")
	res1, fimp1, cmatrix, ypred_proba, bestmodel, perf, auc_score = test_main(x_test, y_test, predictor, testdf, traindf)
	# Uncomment for test results with feature importance (longer run time)
	##res1, fimp1, cmatrix, ytest, ypred_proba, bestmodel, perf, auc_score = test_main(Xtest, ytest, predictor, testdf, traindf, True)

	print("####################Stacking & Weighted Ensemble Testing###########################")
	res2, fimp2, cmatrixstacked, aucstacked = test_stack(x_test, y_test, predstack, testdf, traindf)
	# With feature importance
	##res2, fimp2, cmatrixstacked, aucstacked = test_stack(Xtest, ytest, predstack, testdf, traindf, True)

	return [res1, res2, fimp1, fimp2, cmatrix, bestmodel, perf, auc_score]

In [ ]:
foldtotal = 10
gw_fts_low_path = '/content/features_lim_low_gw.csv'
gw_fts_high_path = '/content/features_lim_high_gw.csv'
gw_fts_medium_path = '/content/features_lim_medium_gw.csv'
normal_fts_medium_path = '/content/features_lim_medium_nrml.csv'
normal_fts_low_path = '/content/features_lim_low_nrml.csv'
normal_fts_high_path='/content/features_lim_high_nrml.csv'
gw_feats_eval_path='/content/features_lim_eval_gw.csv'
normal_feats_eval_path='/content/features_lim_eval_nrml.csv'

gw_feats_low=pd.read_csv(gw_fts_low_path)
gw_feats_medium=pd.read_csv(gw_fts_medium_path)
gw_feats_high=pd.read_csv(gw_fts_high_path)
normal_feats_low=pd.read_csv(normal_fts_low_path)
normal_feats_medium=pd.read_csv(normal_fts_medium_path)
normal_feats_high=pd.read_csv(normal_fts_high_path)
normal_feats = pd.concat([normal_feats_low, normal_feats_high,normal_feats_medium], ignore_index=True)
normal_feats = shuffle(normal_feats, random_state=42)
normal_feats.reset_index(drop=True, inplace=True)
normal_feats['target']=0
gw_feats = pd.concat([gw_feats_low, gw_feats_high,gw_feats_medium], ignore_index=True)
gw_feats = shuffle(gw_feats, random_state=42)
gw_feats.reset_index(drop=True, inplace=True)
gw_feats['target']=1
train=pd.concat([gw_feats,normal_feats],ignore_index=True)


gw_feats_eval=pd.read_csv(gw_feats_eval_path)
gw_feats_eval['target'] = 1

normal_feats_eval=pd.read_csv(normal_feats_eval_path)
normal_feats_eval['target'] = 0
eval_data=pd.concat([gw_feats_eval,normal_feats_eval])
test=eval_data.sample(frac=1, random_state=42)
[ag_res1, ag_res2, fimp1, fimp2, cmatrix, perf, aucscore] = main_ag(train, test, "target")
ff = open("./BinaryTraining.score", "w+")
output_avg(foldtotal, ag_res1, ag_res2, fimp1, fimp2, cmatrix, bestmodel, perf, aucscore, ff)

Verbosity: 4 (Maximum Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
GPU Count:          0
Memory Avail:       9.99 GB / 12.67 GB (78.8%)
Disk Space Avail:   73.76 GB / 107.72 GB (68.5%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='high'         : Strong accuracy wit

[1]	valid_set's binary_logloss: 0.383228	valid_set's balanced_accuracy: 0.5
[2]	valid_set's binary_logloss: 0.349269	valid_set's balanced_accuracy: 0.5
[3]	valid_set's binary_logloss: 0.322509	valid_set's balanced_accuracy: 0.5
[4]	valid_set's binary_logloss: 0.298749	valid_set's balanced_accuracy: 0.5
[5]	valid_set's binary_logloss: 0.279323	valid_set's balanced_accuracy: 0.5
[6]	valid_set's binary_logloss: 0.260968	valid_set's balanced_accuracy: 0.5
[7]	valid_set's binary_logloss: 0.245142	valid_set's balanced_accuracy: 0.5
[8]	valid_set's binary_logloss: 0.230379	valid_set's balanced_accuracy: 0.5
[9]	valid_set's binary_logloss: 0.216845	valid_set's balanced_accuracy: 0.5
[10]	valid_set's binary_logloss: 0.204116	valid_set's balanced_accuracy: 0.715969
[11]	valid_set's binary_logloss: 0.193375	valid_set's balanced_accuracy: 0.874346
[12]	valid_set's binary_logloss: 0.18352	valid_set's balanced_accuracy: 0.896597
[13]	valid_set's binary_logloss: 0.173962	valid_set's balanced_accuracy

Saving /content/AGmodels/models/LightGBMXT/model.pkl
Saving /content/AGmodels/utils/attr/LightGBMXT/y_pred_proba_val.pkl
	0.9961	 = Validation score   (balanced_accuracy)
	6.84s	 = Training   runtime
	0.03s	 = Validation runtime
	77733.9	 = Inference  throughput (rows/s | 2500 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: LightGBM ...
	Fitting LightGBM with 'num_gpus': 0, 'num_cpus': 1
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05}


[1]	valid_set's binary_logloss: 0.380808	valid_set's balanced_accuracy: 0.5
[2]	valid_set's binary_logloss: 0.345093	valid_set's balanced_accuracy: 0.5
[3]	valid_set's binary_logloss: 0.316044	valid_set's balanced_accuracy: 0.5
[4]	valid_set's binary_logloss: 0.291351	valid_set's balanced_accuracy: 0.5
[5]	valid_set's binary_logloss: 0.270127	valid_set's balanced_accuracy: 0.5
[6]	valid_set's binary_logloss: 0.251341	valid_set's balanced_accuracy: 0.5
[7]	valid_set's binary_logloss: 0.234777	valid_set's balanced_accuracy: 0.5
[8]	valid_set's binary_logloss: 0.219889	valid_set's balanced_accuracy: 0.5
[9]	valid_set's binary_logloss: 0.206389	valid_set's balanced_accuracy: 0.5
[10]	valid_set's binary_logloss: 0.194051	valid_set's balanced_accuracy: 0.827225
[11]	valid_set's binary_logloss: 0.182766	valid_set's balanced_accuracy: 0.969895
[12]	valid_set's binary_logloss: 0.172404	valid_set's balanced_accuracy: 0.980366
[13]	valid_set's binary_logloss: 0.162826	valid_set's balanced_accurac

Saving /content/AGmodels/models/LightGBM/model.pkl
Saving /content/AGmodels/utils/attr/LightGBM/y_pred_proba_val.pkl
	0.9945	 = Validation score   (balanced_accuracy)
	4.88s	 = Training   runtime
	0.02s	 = Validation runtime
	116022.4	 = Inference  throughput (rows/s | 2500 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: RandomForestGini ...


...

Saving /content/AGmodels/models/XGBoost/model.pkl
Saving /content/AGmodels/utils/attr/XGBoost/y_pred_proba_val.pkl
	0.9958	 = Validation score   (balanced_accuracy)
	4.38s	 = Training   runtime
	0.03s	 = Validation runtime
	83250.0	 = Inference  throughput (rows/s | 2500 batch size)
Saving /content/AGmodels/models/trainer.pkl
Fitting model: NeuralNetTorch ...
	Fitting NeuralNetTorch with 'num_gpus': 0, 'num_cpus': 1
Tabular Neural Network treats features as the following types:
{
    "continuous": [
        "most_common_dst_port",
        "mean_total_pkts",
        "mean_bytes_recv",
        "median_bytes_recv",
        "mode_bytes_recv",
        "nb_pkts_in",
        "nb_pkts_out",
        "nb_pkts_in_f30",
        "nb_pkts_out_f30",
        "nb_pkts_in_l30",
        "nb_pkts_out_l30",
        "std_pkt_conc_out20",
        "avg_pkt_conc_out20",
        "avg_order_in",
        "avg_order_out",
        "std_order_in",
        "std_order_out",
        "maxconc",
        "perc_in",
      

[1]	valid_set's binary_logloss: 0.398507	valid_set's balanced_accuracy: 0.5
[2]	valid_set's binary_logloss: 0.374134	valid_set's balanced_accuracy: 0.5
[3]	valid_set's binary_logloss: 0.353037	valid_set's balanced_accuracy: 0.5
[4]	valid_set's binary_logloss: 0.334324	valid_set's balanced_accuracy: 0.5
[5]	valid_set's binary_logloss: 0.317414	valid_set's balanced_accuracy: 0.5
[6]	valid_set's binary_logloss: 0.302021	valid_set's balanced_accuracy: 0.5
[7]	valid_set's binary_logloss: 0.288005	valid_set's balanced_accuracy: 0.5
[8]	valid_set's binary_logloss: 0.275252	valid_set's balanced_accuracy: 0.5
[9]	valid_set's binary_logloss: 0.263331	valid_set's balanced_accuracy: 0.5
[10]	valid_set's binary_logloss: 0.25223	valid_set's balanced_accuracy: 0.5
[11]	valid_set's binary_logloss: 0.241875	valid_set's balanced_accuracy: 0.5
[12]	valid_set's binary_logloss: 0.232135	valid_set's balanced_accuracy: 0.5
[13]	valid_set's binary_logloss: 0.223144	valid_set's balanced_accuracy: 0.5
[14]	vali

Saving /content/AGmodels/models/LightGBMLarge/model.pkl
Saving /content/AGmodels/utils/attr/LightGBMLarge/y_pred_proba_val.pkl
	0.9906	 = Validation score   (balanced_accuracy)
	3.67s	 = Training   runtime
	0.01s	 = Validation runtime
	207294.0	 = Inference  throughput (rows/s | 2500 batch size)
Saving /content/AGmodels/models/trainer.pkl
Loading: /content/AGmodels/utils/attr/KNeighborsUnif/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/ExtraTreesGini/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/XGBoost/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/RandomForestGini/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/LightGBM/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/CatBoost/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/NeuralNetFastAI/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/ExtraTreesEntr/y_pred_proba_val.pkl
Loading: /content/AGmodels/utils/attr/NeuralNetTorch/y_pred_proba_val.pkl
Loading: 

[87]	valid_set's binary_logloss: 0.0255632	valid_set's balanced_accuracy: 0.990602
[88]	valid_set's binary_logloss: 0.0250212	valid_set's balanced_accuracy: 0.990602
[89]	valid_set's binary_logloss: 0.0244804	valid_set's balanced_accuracy: 0.990602
[90]	valid_set's binary_logloss: 0.0239532	valid_set's balanced_accuracy: 0.990602


Ensemble size: 1
Ensemble indices: [11]
Ensemble weights: 
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
Saving /content/AGmodels/models/WeightedEnsemble_L2/utils/oof.pkl
Saving /content/AGmodels/models/WeightedEnsemble_L2/model.pkl
	Ensemble Weights: {'NeuralNetTorch': 1.0}
	1.0	 = Validation score   (balanced_accuracy)
	0.27s	 = Training   runtime
	0.0s	 = Validation runtime
	42680.6	 = Inference  throughput (rows/s | 2500 batch size)
Saving /content/AGmodels/models/trainer.pkl
Saving /content/AGmodels/models/trainer.pkl
Saving /content/AGmodels/models/trainer.pkl
AutoGluon training complete, total runtime = 687.16s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 42680.6 rows/s (2500 batch size)
Loading: /content/AGmodels/models/trainer.pkl
Enabling decision threshold calibration (calibrate_decision_threshold='auto', metric is valid, problem_type is 'binary')
Loading: /content/AGmodels/utils/data/X_val.pkl
Loading: /content/AGmodels/utils/data/y_val.pkl
Loading: /conten